# Week 1: Data Loading and Exploration

**Project:** How Data-Quality Defects Affect Credit-Risk Models and Their Explanations

**Goal this week:** load the dataset, understand it, document its existing quality issues, and save a clean baseline that every later experiment starts from.

**Dataset:** *Default of Credit Card Clients* (Yeh & Lien, 2009), UCI ML Repository id 350. 30,000 credit-card clients in Taiwan, 23 features, binary target: did the client default the next month?

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Works whether you open this notebook from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_raw, clean, FEATURES, TARGET, PAY_STATUS

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 110

## 2. Load the raw data

`load_raw` first looks for a file in `data/raw/`. If there is none, it downloads the data with `ucimlrepo` and caches it.

In [ ]:
raw = load_raw(ROOT / "data" / "raw")
print(raw.shape)
raw.head()

## 3. First checks: types, missing values, duplicates

In [ ]:
print("Rows, columns:", raw.shape)
print("Missing values in total:", int(raw.isna().sum().sum()))
print("Exact duplicate rows:", int(raw.duplicated().sum()))
raw.describe().T

## 4. Target balance

Most clients do not default, so the classes are imbalanced. This matters later: plain accuracy would be misleading, which is why the project uses AUC and the Brier score.

In [ ]:
rate = raw[TARGET].mean()
print(f"Default rate: {rate:.1%}  ({raw[TARGET].sum():,} of {len(raw):,} clients)")

ax = raw[TARGET].value_counts().sort_index().plot.bar(color=["#4C72B0", "#C44E52"])
ax.set_xticklabels(["No default (0)", "Default (1)"], rotation=0)
ax.set_ylabel("Clients"); ax.set_title("Target balance")
plt.tight_layout(); plt.savefig(FIG_DIR / "w1_target_balance.png"); plt.show()

## 5. Existing data-quality issues in a "clean" benchmark

This section is important for the research story. The dataset is widely treated as clean, but several columns contain codes that the official documentation does not define:

- **EDUCATION**: documented 1–4, but 0, 5 and 6 also appear.
- **MARRIAGE**: documented 1–3, but 0 also appears.
- **PAY_0 … PAY_6**: documented −1 and 1–9, but −2 and 0 also appear, very often.

In [ ]:
for col in ["EDUCATION", "MARRIAGE"]:
    print(col, raw[col].value_counts().sort_index().to_dict())

pay_codes = pd.DataFrame({c: raw[c].value_counts() for c in PAY_STATUS}).fillna(0).astype(int).sort_index()
pay_codes

## 6. Clean baseline

Decisions (all logged by `clean`):
- Undocumented EDUCATION codes are merged into 4 (*others*); undocumented MARRIAGE codes into 3 (*others*).
- PAY codes −2 and 0 are **kept**. They are frequent and predictive, and removing them would discard most of the data.
- Nothing is dropped or imputed, so the baseline stays as close to the original as possible.

In [ ]:
df, log = clean(raw)
for line in log:
    print("-", line)

## 7. Who defaults? Default rate by key variables

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

# Most recent repayment status
g = df.groupby("PAY_0")[TARGET].mean()
axes[0].bar(g.index.astype(str), g.values, color="#4C72B0")
axes[0].set_title("By most recent repayment status (PAY_0)")
axes[0].set_xlabel("PAY_0 (months delayed; -2/-1/0 = on time or no use)")

# Age group
age_grp = pd.cut(df["AGE"], [20, 30, 40, 50, 60, 80])
g = df.groupby(age_grp, observed=True)[TARGET].mean()
axes[1].bar(g.index.astype(str), g.values, color="#55A868")
axes[1].set_title("By age group")

# Credit limit quintile
lim_q = pd.qcut(df["LIMIT_BAL"], 5)
g = df.groupby(lim_q, observed=True)[TARGET].mean()
axes[2].bar(range(5), g.values, color="#C44E52")
axes[2].set_xticks(range(5)); axes[2].set_xticklabels(["Q1\n(lowest)", "Q2", "Q3", "Q4", "Q5\n(highest)"])
axes[2].set_title("By credit-limit quintile")

for ax in axes:
    ax.set_ylabel("Default rate"); ax.axhline(df[TARGET].mean(), ls="--", c="gray", lw=1)
plt.tight_layout(); plt.savefig(FIG_DIR / "w1_default_rates.png"); plt.show()

Groups whose default rates differ sharply (for example low vs high credit limits) are natural candidates for the **distribution-shift experiment** in Week 4: train on one group, test on another.

## 8. Distributions of key numeric features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
df["LIMIT_BAL"].plot.hist(bins=50, ax=axes[0], color="#4C72B0", title="Credit limit (NT$)")
df["AGE"].plot.hist(bins=40, ax=axes[1], color="#55A868", title="Age")
df["BILL_AMT1"].plot.hist(bins=60, ax=axes[2], color="#8172B2", title="Most recent bill amount (NT$)")
plt.tight_layout(); plt.savefig(FIG_DIR / "w1_distributions.png"); plt.show()

print("Negative bill amounts (credit balances):", int((df[[f'BILL_AMT{i}' for i in range(1,7)]] < 0).any(axis=1).sum()), "clients")

## 9. Correlation with the target

In [ ]:
corr = df.corr(numeric_only=True)[TARGET].drop(TARGET).sort_values()
ax = corr.plot.barh(figsize=(6, 7), color=np.where(corr > 0, "#C44E52", "#4C72B0"))
ax.set_title("Correlation of each feature with default"); ax.axvline(0, c="k", lw=0.8)
plt.tight_layout(); plt.savefig(FIG_DIR / "w1_target_correlation.png"); plt.show()

Repayment-status features (PAY_*) are expected to be the strongest signals. In Week 2, check whether SHAP agrees. That clean-data ranking becomes the reference point for measuring explanation stability.

## 10. Save the clean baseline

In [ ]:
out = ROOT / "data" / "processed" / "credit_clean.csv"
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)
print(f"Saved {df.shape} to {out.relative_to(ROOT)}")

## 11. Week 1 findings (write these in your own words)

Fill this in before moving on. These notes become the "Data" section of your final report.

1. **Size and balance:** …
2. **Undocumented codes found:** …
3. **Cleaning decisions and why:** …
4. **Strongest signals of default:** …
5. **Candidate groups for the distribution-shift experiment:** …
6. **Questions or surprises:** …